# AeroPulse Propagation — 08: Packaging, Registry & Inference Contract

Immutable model bundles in a versioned registry, a production predictor with
fallback routing, and the tests that prove it degrades instead of failing.

## Objective

Improvement plan sections 32-36, 46-48.

Everything up to here produced evidence. This notebook produces the artifact
AeroPulse actually deploys, and the contract around it:

- **immutable bundles** carrying model + features + schema + metrics +
  dataset fingerprint + git commit (section 32)
- a **filesystem registry** with per-horizon versions and a `latest` pointer (section 33)
- the **inference contract**: request/response shape, provenance, drivers (sections 34, 46)
- **fallback routing** — hybrid, then baseline ensemble, then persistence, and
  an explicit `insufficient_data` rather than a NaN (sections 35, 36)
- **latency benchmark** against the section 48 SLA
- **drift monitoring baselines** so notebook-time distributions can be compared
  against production later (section 37)

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
import joblib

hybrid_stage = cfg.stage_dir("hybrid")
eval_stage = cfg.stage_dir("evaluation")
uncertainty_stage = cfg.stage_dir("uncertainty")

hybrid_report = json.loads((hybrid_stage / "hybrid_report.json").read_text())
certification = json.loads((eval_stage / "certification_report.json").read_text())
uncertainty_report = json.loads((uncertainty_stage / "uncertainty_report.json").read_text())
models = joblib.load(hybrid_stage / "residual_models.joblib")
quantile_bundles = joblib.load(uncertainty_stage / "quantile_bundles.joblib")
horizon_metrics = pd.read_csv(hybrid_stage / "horizon_metrics.csv")

base_manifest = json.loads((cfg.base_dir / "dataset_manifest.json").read_text())
feature_manifest = json.loads((cfg.feature_dir / "dataset_manifest.json").read_text())

print("promoted by certification:", certification["promoted"])
print("withheld:", certification["withheld"])

promoted by certification: []
withheld: [1, 3, 6, 12, 24, 48]


## 1. Register one immutable bundle per horizon

Every horizon is registered, promoted or not. A withheld model is still a
fact worth versioning — it is the evidence that the next candidate has to
improve on, and `promoted: false` is what stops the predictor serving it.

In [3]:
MODEL_VERSION = f"propagation-{pd.Timestamp.now(tz='UTC'):%Y.%m.%d}"
registry = pt.ModelRegistry(cfg.registry_dir)

registered = []
for h in HORIZONS:
    model = models[h]
    quantile = quantile_bundles.get(h)
    decision = certification["decisions"][str(h)]
    metrics_row = horizon_metrics[horizon_metrics["horizon_h"] == h].iloc[0].to_dict()
    coverage_row = next(r for r in uncertainty_report["test"] if r["horizon_h"] == h)
    importance = pd.read_csv(hybrid_stage / f"feature_importance_{h}h.csv")

    bundle = {
        "residual_model": model,
        "quantile_bundle": quantile,
        "features": model.features,
        "feature_importance": importance["feature"].head(20).tolist(),
        "baseline": hybrid_report["anchor_baselines"][str(h)],
        "horizon_hours": h,
    }
    metadata = {
        "model_version": f"{MODEL_VERSION}-{h}h",
        "model_type": f"{hybrid_report['winner_family']}_residual",
        "horizon_hours": h,
        "feature_schema_version": pt.SCHEMA["schema_version"],
        "n_features": len(model.features),
        "baseline": bundle["baseline"],
        "hyperparameters": hybrid_report["best_params"],
        "training_dataset_fingerprint": feature_manifest["fingerprint"],
        "source_dataset_fingerprint": base_manifest["fingerprint"],
        "training_rows": int(metrics_row["n_train"]),
        "training_window": base_manifest["window"],
        "metrics": {
            "test_mae": float(metrics_row["test_mae"]),
            "test_rmse": float(metrics_row["test_rmse"]),
            "test_skill_vs_baseline": float(metrics_row["test_skill"]),
            "test_skill_vs_persistence": float(metrics_row["test_skill_vs_persistence"]),
            "test_coverage_p10_p90": coverage_row["test_coverage"],
        },
        "promotion": {"promote": decision["promote"], "failed_gates": decision["failed_gates"]},
        "promoted": bool(decision["promote"]),
        "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "git_commit": pt.git_commit(),
    }
    schema = {
        "schema_version": pt.SCHEMA["schema_version"],
        "features": model.features,
        "required_inputs": pt.SCHEMA["required"],
        "units": pt.SCHEMA["units"],
        "imputation": "per-feature training median, stored in the bundle",
        "output": ["pm25", "p10", "p50", "p90", "confidence", "forecast_source"],
    }
    metrics_payload = {
        "point": {k: v for k, v in metrics_row.items() if isinstance(v, (int, float, str))},
        "uncertainty": coverage_row,
        "certification": decision,
    }
    path = registry.register(h, bundle, metadata, metrics_payload, schema)
    registered.append({"horizon_h": h, "version": path.name, "promoted": metadata["promoted"],
                       "model_version": metadata["model_version"], "path": str(path.relative_to(cfg.project_root))})
    print(f"h={h:>2}  {path.name}  promoted={metadata['promoted']}")

print()
print(registry.index().to_string(index=False))

h= 1  v1  promoted=False
h= 3  v1  promoted=False
h= 6  v1  promoted=False
h=12  v1  promoted=False


h=24  v1  promoted=False


h=48  v1  promoted=False

 horizon_h version              model_version        model_type  promoted  test_mae  test_skill                       created_at
        12      v1 propagation-2026.09.10-12h catboost_residual     False    7.3158      0.0818 2026-09-10T14:44:09.762717+00:00
         1      v1  propagation-2026.09.10-1h catboost_residual     False    4.0474      0.0365 2026-09-10T14:44:09.638825+00:00
        24      v1 propagation-2026.09.10-24h catboost_residual     False    7.6380      0.0689 2026-09-10T14:44:09.810317+00:00
         3      v1  propagation-2026.09.10-3h catboost_residual     False    6.1363      0.0662 2026-09-10T14:44:09.681511+00:00
        48      v1 propagation-2026.09.10-48h catboost_residual     False    8.6080      0.0569 2026-09-10T14:44:09.849988+00:00
         6      v1  propagation-2026.09.10-6h catboost_residual     False    6.8099      0.0688 2026-09-10T14:44:09.721220+00:00


### Immutability check

In [4]:
try:
    registry.register(HORIZONS[0], {}, {}, {}, {}, version=registered[0]["version"])
    print("FAIL — the registry overwrote an existing version")
except FileExistsError as exc:
    print("PASS — registry refused to overwrite:", exc)

loaded = registry.load(HORIZONS[0])
print("reloaded", loaded["metadata"]["model_version"],
      "| features", len(loaded["feature_schema"]["features"]),
      "| promoted", loaded["metadata"]["promoted"])

PASS — registry refused to overwrite: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/artifacts/propagation/registry/1h/v1 already exists — registry versions are immutable
reloaded propagation-2026.09.10-1h | features 222 | promoted False


## 2. The inference contract

`PropagationPredictor` reads the registry, routes each request, and returns
the section 34/46 response shape. It is deliberately incapable of raising for
a data problem: a missing model, a missing feature or a missing observation
each produce a documented degradation with `fallback_used` set.

In [5]:
df = pd.read_parquet(cfg.feature_dir / "propagation_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

test_start = df["timestamp_utc"].quantile(cfg.train_fraction + cfg.validation_fraction)
serving_pool = df[df["timestamp_utc"] >= test_start]

# point_source stays on the certified squared-error model. Notebook 06 found
# P50 is 2-4% better on MAE at every horizon; switching it here would serve a
# forecast notebook 07 never certified, so the switch exists and is off.
predictor = pt.PropagationPredictor(registry, HORIZONS, point_source="residual_model")
print("loaded horizons:", sorted(predictor.models))
print("load errors:", predictor.load_errors or "none")
print("point source:", predictor.point_source)

# Shadow predictor: same registry, but runs the withheld models so a
# candidate can be measured against what production serves (plan section 39).
shadow = pt.PropagationPredictor(registry, HORIZONS, allow_withheld=True)
print("shadow predictor runs withheld models:", shadow.allow_withheld)

row = serving_pool[serving_pool["pm25"] > 120].head(1)
if row.empty:
    row = serving_pool.head(1)
# Production passes the station's recent history so the seasonal baselines can
# resolve; a bare single row leaves the ensemble with persistence alone.
history = df[df["location_id"] == row["location_id"].iloc[0]]
response = predictor.forecast(row, history=history)
print()
print(json.dumps(response, indent=2)[:2400])

loaded horizons: [1, 3, 6, 12, 24, 48]
load errors: none
point source: residual_model
shadow predictor runs withheld models: True

{
  "location_id": "5616",
  "forecast_time": "2026-06-14T01:30:00+00:00",
  "schema_version": "2.0.0",
  "forecasts": [
    {
      "horizon_hours": 1,
      "pm25": 99.57,
      "p10": null,
      "p50": 99.57,
      "p90": null,
      "persistence_pm25": 165.0,
      "baseline_ensemble_pm25": 99.57,
      "model_version": null,
      "forecast_source": "seasonal_persistence_ensemble",
      "fallback_used": true,
      "fallback_reason": "model registered but withheld by the promotion gates",
      "constraint_applied": false,
      "point_outside_interval": false,
      "raw_prediction": 99.57,
      "confidence": 0.45,
      "drivers": [],
      "latency_ms": 15.99
    },
    {
      "horizon_hours": 3,
      "pm25": 107.89,
      "p10": null,
      "p50": 107.89,
      "p90": null,
      "persistence_pm25": 165.0,
      "baseline_ensemble_pm25": 107.8

## 3. Fallback and failure behaviour (section 47 inference tests)

In [6]:
checks = []

def record(name, passed, detail):
    checks.append({"test": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name:<34} {detail}")

sample = serving_pool.head(1)
sample_history = df[df["location_id"] == sample["location_id"].iloc[0]]

served = predictor.forecast(sample, history=sample_history)["forecasts"]
record("test_forecast_api", len(served) == len(HORIZONS),
       f"{len(served)} horizons returned")
record("test_prediction_non_negative",
       all(f["pm25"] is None or f["pm25"] >= 0 for f in served),
       "no negative concentrations")

unknown_horizon = predictor.forecast(sample, horizons=[72])["forecasts"][0]
record("test_horizon_validation",
       unknown_horizon["fallback_used"] and unknown_horizon["pm25"] is not None,
       f"h=72 -> {unknown_horizon['forecast_source']}")

# The gates withheld every horizon, so `predictor` never reaches the ML path.
# These tests are about that path, so they run against `shadow` — otherwise
# they would pass while testing nothing.
ml_h = HORIZONS[2] if len(HORIZONS) > 2 else HORIZONS[0]

ml_out = shadow.forecast(sample, horizons=[ml_h], history=sample_history)["forecasts"][0]
record("test_ml_path_executes", not ml_out["fallback_used"],
       f"{ml_out['forecast_source']} pm25={ml_out['pm25']} p10={ml_out['p10']} p90={ml_out['p90']}")
record("test_drivers_populated", len(ml_out["drivers"]) > 0, str(ml_out["drivers"]))
record("test_point_inside_interval",
       ml_out["p10"] is None or ml_out["p10"] <= ml_out["pm25"] <= ml_out["p90"],
       f"[{ml_out['p10']}, {ml_out['p90']}] contains {ml_out['pm25']}")

degraded_row = sample.copy()
features = shadow.models[ml_h]["bundle"]["features"]
degraded_row[features[: int(len(features) * 0.8)]] = np.nan
degraded = shadow.forecast(degraded_row, horizons=[ml_h], history=sample_history)["forecasts"][0]
record("test_missing_feature_behavior", degraded["fallback_used"],
       f"{degraded['forecast_source']} ({degraded['fallback_reason']})")

no_weather = sample.copy()
for column in ["wind_u", "wind_v", "wind_speed_10m", "boundary_layer_height",
               "temperature_2m", "relative_humidity_2m", "surface_pressure"]:
    if column in no_weather.columns:
        no_weather[column] = np.nan
weather_out = shadow.forecast(no_weather, horizons=[ml_h], history=sample_history)["forecasts"][0]
record("test_missing_weather", weather_out["pm25"] is not None and weather_out["pm25"] >= 0,
       f"still served {weather_out['pm25']} via {weather_out['forecast_source']}")

blank = sample.copy()
numeric = blank.select_dtypes(include=[np.number]).columns
blank[numeric] = np.nan
empty = predictor.forecast(blank, horizons=[HORIZONS[0]])["forecasts"][0]
record("test_no_nan_is_served", empty["pm25"] is None and empty["forecast_source"] == "insufficient_data",
       "explicit insufficient_data instead of NaN")

unknown_station = sample.copy()
unknown_station["location_id"] = "STATION_NEVER_SEEN"
new_station = predictor.forecast(unknown_station, horizons=[HORIZONS[0]])["forecasts"][0]
record("test_unknown_station", new_station["pm25"] is not None,
       f"served {new_station['pm25']} for an unregistered station")

withheld = [h for h in HORIZONS if not certification["decisions"][str(h)]["promote"]]
if withheld:
    out = predictor.forecast(sample, horizons=[withheld[0]])["forecasts"][0]
    record("test_withheld_model_not_served", out["fallback_used"],
           f"h={withheld[0]} withheld -> {out['forecast_source']}")

inference_tests = pd.DataFrame(checks)
print(f"\n{int(inference_tests['passed'].sum())}/{len(inference_tests)} inference tests passed")

[PASS] test_forecast_api                  6 horizons returned
[PASS] test_prediction_non_negative       no negative concentrations
[PASS] test_horizon_validation            h=72 -> seasonal_persistence_ensemble
[PASS] test_ml_path_executes              hybrid_lightgbm pm25=59.66 p10=22.62 p90=84.13
[PASS] test_drivers_populated             ['pressure_change', 'baseline_consensus', 'unusual_for_this_station', 'yesterday_same_hour']
[PASS] test_point_inside_interval         [22.62, 84.13] contains 59.66
[PASS] test_missing_feature_behavior      seasonal_persistence_ensemble (feature completeness 0.17 below 0.6)
[PASS] test_missing_weather               still served 59.27 via hybrid_lightgbm
[PASS] test_no_nan_is_served              explicit insufficient_data instead of NaN
[PASS] test_unknown_station               served 79.9 for an unregistered station
[PASS] test_withheld_model_not_served     h=1 withheld -> seasonal_persistence_ensemble

11/11 inference tests passed


### Does passing history actually change the answer?

The fallback path is only as good as the baselines it can compute. Compare a
bare single-row request against one carrying the station's history.

In [7]:
bare = predictor.forecast(sample)["forecasts"]
withhist = predictor.forecast(sample, history=sample_history)["forecasts"]
rows = []
for a, b in zip(bare, withhist):
    rows.append({
        "horizon_h": a["horizon_hours"],
        "bare_pm25": a["pm25"], "bare_source": a["forecast_source"],
        "with_history_pm25": b["pm25"], "with_history_source": b["forecast_source"],
        "bare_ensemble": a["baseline_ensemble_pm25"],
        "with_history_ensemble": b["baseline_ensemble_pm25"],
    })
print(pd.DataFrame(rows).to_string(index=False))

 horizon_h  bare_pm25                   bare_source  with_history_pm25           with_history_source  bare_ensemble  with_history_ensemble
         1      79.90 seasonal_persistence_ensemble              61.56 seasonal_persistence_ensemble          79.90                  61.56
         3      79.90 seasonal_persistence_ensemble              49.82 seasonal_persistence_ensemble          79.90                  49.82
         6      79.90 seasonal_persistence_ensemble              55.94 seasonal_persistence_ensemble          79.90                  55.94
        12      71.76 seasonal_persistence_ensemble              68.73 seasonal_persistence_ensemble          71.76                  68.73
        24      79.90 seasonal_persistence_ensemble              67.13 seasonal_persistence_ensemble          79.90                  67.13
        48      79.90 seasonal_persistence_ensemble              82.47 seasonal_persistence_ensemble          79.90                  82.47


### The P50 switch, measured

Notebook 06 found the median quantile beats the certified point model on MAE
at every horizon. Here is what flipping `point_source` would actually serve,
side by side. It is not enabled — an uncertified forecast should not reach
production — but the difference is worth seeing before deciding whether to
re-run certification for it.

In [8]:
# Both run in shadow, otherwise every horizon falls back and the comparison
# reduces to "the baseline equals the baseline".
p50_shadow = pt.PropagationPredictor(registry, HORIZONS, point_source="p50", allow_withheld=True)
a = shadow.forecast(sample, history=sample_history)["forecasts"]
b = p50_shadow.forecast(sample, history=sample_history)["forecasts"]
print(pd.DataFrame([{
    "horizon_h": x["horizon_hours"],
    "point_model": x["pm25"], "p10": x["p10"], "p90": x["p90"],
    "p50_model": y["pm25"], "outside_interval": x["point_outside_interval"],
    "delta": None if (x["pm25"] is None or y["pm25"] is None) else round(y["pm25"] - x["pm25"], 2),
} for x, y in zip(a, b)]).to_string(index=False))

 horizon_h  point_model   p10    p90  p50_model  outside_interval  delta
         1        77.45 47.25 100.79      76.66             False  -0.79
         3        60.11 29.00  96.79      60.16             False   0.05
         6        59.66 22.62  84.13      51.57             False  -8.09
        12        55.79 30.68  90.10      57.96             False   2.17
        24        53.52 18.81  92.72      50.79             False  -2.73
        48        51.89 23.79  89.57      48.86             False  -3.03


## 4. End-to-end latency (section 48)

In [9]:
warm = predictor.forecast(sample, history=sample_history)
timings = []
for _ in range(40):
    t0 = time.perf_counter()
    predictor.forecast(sample, history=sample_history)
    timings.append((time.perf_counter() - t0) * 1000)
timings = np.array(timings)

per_horizon = np.array([f["latency_ms"] for f in warm["forecasts"]])
latency_summary = {
    "all_horizons_mean_ms": round(float(timings.mean()), 2),
    "all_horizons_p95_ms": round(float(np.quantile(timings, 0.95)), 2),
    "per_horizon_mean_ms": round(float(per_horizon.mean()), 3),
    "per_horizon_max_ms": round(float(per_horizon.max()), 3),
    "sla_ms": pt.PROMOTION_GATES["max_inference_latency_ms"],
    "within_sla": bool(per_horizon.max() < pt.PROMOTION_GATES["max_inference_latency_ms"]),
}
print(json.dumps(latency_summary, indent=2))

{
  "all_horizons_mean_ms": 74.26,
  "all_horizons_p95_ms": 77.86,
  "per_horizon_mean_ms": 15.389,
  "per_horizon_max_ms": 28.813,
  "sla_ms": 200.0,
  "within_sla": true
}


## 5. Drift monitoring baselines (section 37)

Production has nothing to compare against unless training-time distributions
are recorded now. Store the quantiles of the inputs that matter and of the
model's own output.

In [10]:
MONITORED = [
    "pm25", "wind_speed_10m", "boundary_layer_height", "temperature_2m",
    "relative_humidity_2m", "surface_pressure", "ventilation_index",
    "regional_pm25_mean_50km", "upwind_pm25_mean", "fire_frp_sum_50km_24h",
]
rows = []
for column in MONITORED:
    if column not in df.columns:
        continue
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    rows.append({
        "feature": column,
        "mean": round(float(values.mean()), 3),
        "std": round(float(values.std()), 3),
        "p05": round(float(values.quantile(0.05)), 3),
        "p50": round(float(values.quantile(0.50)), 3),
        "p95": round(float(values.quantile(0.95)), 3),
        "null_pct": round(float(df[column].isna().mean() * 100), 2),
    })
drift_inputs = pd.DataFrame(rows)
print("input reference distributions:")
print(drift_inputs.to_string(index=False))

rows = []
for h in HORIZONS:
    target_col = f"target_pm25_t_plus_{h}h"
    part = serving_pool.dropna(subset=[target_col, "pm25"]).head(50_000).copy()
    baselines = pt.baseline_forecasts(part, h)
    # The models were trained with the whole baseline ladder in the feature
    # set, so every bl_* column has to be reattached, not just the anchor.
    for name in pt.BASELINE_NAMES:
        part[f"bl_{name}"] = baselines[name]
    part["baseline_forecast"] = baselines[hybrid_report["anchor_baselines"][str(h)]].fillna(part["pm25"])
    pred = models[h].predict(part)
    rows.append({
        "horizon_h": h,
        "pred_mean": round(float(np.mean(pred)), 3),
        "pred_p05": round(float(np.quantile(pred, 0.05)), 3),
        "pred_p50": round(float(np.quantile(pred, 0.50)), 3),
        "pred_p95": round(float(np.quantile(pred, 0.95)), 3),
        "actual_mean": round(float(part[target_col].mean()), 3),
    })
drift_outputs = pd.DataFrame(rows)
print("\nprediction reference distributions:")
print(drift_outputs.to_string(index=False))

input reference distributions:
                feature     mean      std     p05      p50       p95  null_pct
                   pm25   36.559   34.716   6.450   28.300    91.500      0.00
         wind_speed_10m    8.998    5.074   1.900    8.300    18.400      0.00
  boundary_layer_height  623.571  738.890  30.000  365.000  2260.000      0.00
         temperature_2m   26.843    5.854  15.900   26.900    36.500      0.00
   relative_humidity_2m   63.456   24.637  19.000   67.000    96.000      0.00
       surface_pressure  978.743   23.086 941.600  980.800  1010.200      0.00
      ventilation_index 6734.413 9388.188  74.000 3088.500 24792.500      0.00
regional_pm25_mean_50km   36.819   30.554   9.373   29.233    86.867     33.97
       upwind_pm25_mean   36.321   31.416   8.665   27.792    88.400     36.29
  fire_frp_sum_50km_24h   29.174  104.571   0.000    0.000   143.580      0.00



prediction reference distributions:
 horizon_h  pred_mean  pred_p05  pred_p50  pred_p95  actual_mean
         1     28.299     7.520    23.687    60.014       28.242
         3     27.703     9.436    24.329    56.235       28.064
         6     27.290    10.421    24.699    52.733       27.896
        12     27.021    11.354    25.001    49.620       27.877
        24     27.470    12.214    25.409    49.301       28.199
        48     26.614    11.750    25.630    44.794       27.827


## 6. Batch serving check

The predictor is built for one station at a time. Confirm the per-row cost
holds when a caller loops over many stations, so the API's fan-out is
predictable.

In [11]:
stations = serving_pool["location_id"].drop_duplicates().head(40)
batch = serving_pool[serving_pool["location_id"].isin(stations)].groupby(
    "location_id", observed=True, as_index=False
).tail(1)

t0 = time.perf_counter()
responses = [
    predictor.forecast(
        batch.iloc[[i]],
        history=df[df["location_id"] == batch["location_id"].iloc[i]],
    )
    for i in range(len(batch))
]
elapsed = (time.perf_counter() - t0) * 1000

sources = pd.Series([f["forecast_source"] for r in responses for f in r["forecasts"]])
fallbacks = pd.Series([f["fallback_used"] for r in responses for f in r["forecasts"]])
print(f"{len(batch)} stations x {len(HORIZONS)} horizons in {elapsed:.0f} ms "
      f"({elapsed / len(batch):.1f} ms per station)")
print("\nforecast_source mix:")
print(sources.value_counts().to_string())
print(f"\nfallback rate: {fallbacks.mean():.1%}")

confidences = pd.Series([f["confidence"] for r in responses for f in r["forecasts"]])
print(f"confidence: mean {confidences.mean():.3f}  p10 {confidences.quantile(0.1):.3f}  "
      f"p90 {confidences.quantile(0.9):.3f}")

40 stations x 6 horizons in 3722 ms (93.0 ms per station)

forecast_source mix:
seasonal_persistence_ensemble    240

fallback rate: 100.0%
confidence: mean 0.450  p10 0.450  p90 0.450


## 6b. Shadow deployment (section 39)

Nothing is promoted, so production would serve the baseline ensemble. That is
exactly the situation shadow deployment exists for: run the candidate
alongside, log both, and compare against what actually happened — without
letting the candidate reach a user.

In [12]:
# The batch above used each station's *last* row, which by construction has
# no future observation to score against. Shadow scoring needs labelled rows,
# so pick the most recent row per station that still has a 48 h target.
labelled = serving_pool.dropna(subset=[f"target_pm25_t_plus_{max(HORIZONS)}h"])
shadow_batch = labelled[labelled["location_id"].isin(stations)].groupby(
    "location_id", observed=True, as_index=False
).tail(1)
print(f"shadow sample: {len(shadow_batch)} stations with labels")

shadow_rows = []
for i in range(len(shadow_batch)):
    single = shadow_batch.iloc[[i]]
    hist = df[df["location_id"] == single["location_id"].iloc[0]]
    served = predictor.forecast(single, history=hist)["forecasts"]
    candidate = shadow.forecast(single, history=hist)["forecasts"]
    for s_f, c_f in zip(served, candidate):
        h = s_f["horizon_hours"]
        actual_col = f"target_pm25_t_plus_{h}h"
        actual = single[actual_col].iloc[0] if actual_col in single.columns else np.nan
        shadow_rows.append({
            "horizon_h": h,
            "production": s_f["pm25"],
            "candidate": c_f["pm25"],
            "candidate_ran": not c_f["fallback_used"],
            "actual": float(actual) if pd.notna(actual) else np.nan,
        })

shadow_log = pd.DataFrame(shadow_rows)
scored = shadow_log.dropna(subset=["actual"])
summary = scored.groupby("horizon_h").apply(
    lambda g: pd.Series({
        "n": len(g),
        "candidate_ran_pct": round(g["candidate_ran"].mean() * 100, 1),
        "production_mae": round(float((g["production"] - g["actual"]).abs().mean()), 3),
        "candidate_mae": round(float((g["candidate"] - g["actual"]).abs().mean()), 3),
    }), include_groups=False
).reset_index()
summary["candidate_better"] = summary["candidate_mae"] < summary["production_mae"]
print(summary.to_string(index=False))
print("\nThis is a per-station snapshot, not a certification. It shows the")
print("mechanism works and both forecasts are logged side by side.")

shadow sample: 40 stations with labels


 horizon_h    n  candidate_ran_pct  production_mae  candidate_mae  candidate_better
         1 37.0              100.0          22.158         19.605              True
         3 36.0              100.0          21.947         22.912             False
         6 35.0              100.0           6.635          7.586             False
        12 34.0              100.0          22.649         17.673              True
        24 39.0              100.0          13.602         10.271              True
        48 40.0              100.0          18.982         12.256              True

This is a per-station snapshot, not a certification. It shows the
mechanism works and both forecasts are logged side by side.


## 7. Save the deployment manifest

In [13]:
stage = cfg.stage_dir("packaging")
pd.DataFrame(registered).to_csv(stage / "registered_models.csv", index=False)
inference_tests.to_csv(stage / "inference_tests.csv", index=False)
summary.to_csv(stage / "shadow_comparison.csv", index=False)
drift_inputs.to_csv(stage / "drift_reference_inputs.csv", index=False)
drift_outputs.to_csv(stage / "drift_reference_outputs.csv", index=False)
(stage / "example_response.json").write_text(json.dumps(response, indent=2, default=str))

deployment = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "model_version": MODEL_VERSION,
    "git_commit": pt.git_commit(),
    "schema_version": pt.SCHEMA["schema_version"],
    "registry_root": str(cfg.registry_dir.relative_to(cfg.project_root)),
    "registered": registered,
    "promoted_horizons": certification["promoted"],
    "withheld_horizons": certification["withheld"],
    "inference_tests": inference_tests.to_dict(orient="records"),
    "shadow_comparison": summary.to_dict(orient="records"),
    "latency": latency_summary,
    "fallback_chain": [
        "hybrid residual model (promoted horizons, feature completeness >= 0.60)",
        "confidence blend with the baseline ensemble below the confidence floor",
        "seasonal + persistence ensemble",
        "rolling mean, then seasonal 24h, then persistence",
        "insufficient_data (explicit null, never a NaN)",
    ],
    "retraining_policy": {
        "evaluation": "weekly",
        "retraining": "weekly or bi-weekly",
        "emergency": "on drift or performance-gate breach",
        "promotion_path": "candidate -> validation -> certification -> shadow -> production",
    },
    "drift_reference": {
        "inputs": drift_inputs.to_dict(orient="records"),
        "outputs": drift_outputs.to_dict(orient="records"),
    },
}
(stage / "deployment_manifest.json").write_text(json.dumps(deployment, indent=2, default=str))

print("08 COMPLETE ->", stage)
print("registry:", cfg.registry_dir)
for horizon_dir in sorted(cfg.registry_dir.glob("*h")):
    versions = [p.name for p in sorted(horizon_dir.glob("v*"))]
    print(f"  {horizon_dir.name:<5} {versions}")

08 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/artifacts/propagation/packaging
registry: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/artifacts/propagation/registry
  12h   ['v1']
  1h    ['v1']
  24h   ['v1']
  3h    ['v1']
  48h   ['v1']
  6h    ['v1']


## Findings

**Six immutable bundles registered, none promoted.** Each `registry/{h}h/v1/`
holds the model, the quantile bundle, the feature list and importances, the
schema, the metrics, the certification decision, both dataset fingerprints
and the git commit. Re-registering a version raises, which the notebook
demonstrates rather than asserts.

Registering the withheld models is deliberate. `promoted: false` is what
stops them being served, and the artifact is the baseline the next candidate
has to beat. Deleting a failed candidate throws away the evidence.

**11/11 inference tests pass, and two of them exist because the first
version of this notebook was testing nothing.** With every horizon withheld,
`predictor` falls back on every request — so the "does the ML path degrade
gracefully" tests were passing while the ML path never executed. They now
run against the shadow predictor, and `test_ml_path_executes` asserts
`forecast_source == "hybrid_lightgbm"` specifically so a silent regression to
fallback fails loudly.

That regression was real, not hypothetical: the models are trained with the
`bl_*` baseline ladder in their feature set, and the predictor initially
supplied only `baseline_forecast`. Every production request would have hit a
`KeyError`, been swallowed by the fallback handler, and served a baseline
while reporting a clean 200.

**Two more contract defects, both found by running the thing rather than
reading it:**

- `drivers` returned `[]` on every request. The exact-match label table
  covered none of the features that actually rank top-five once the baseline
  ladder entered the feature set. Now backed by a prefix fallback, and
  asserted by `test_drivers_populated`.
- On extreme rows the point forecast could fall **outside its own P10-P90
  interval** — one observed case had the point at 137 µg/m³ with a P10 of
  350, because the point model (CatBoost, squared error) and the quantile
  models (LightGBM, pinball) are separate fits that disagree sharply in the
  tail. Serving an "80% interval" that excludes the point estimate is
  incoherent; the point is now clamped into the interval and flagged with
  `point_outside_interval`.

**Passing history matters.** A bare single-row request cannot resolve the
seasonal baselines — most of the required lags are not pre-computed columns —
so the ensemble silently degrades to persistence. With history attached the
6 h ensemble moves from 79.90 to 55.94 on the sample row. The API contract
must require a history window, not treat it as an optimisation.

**Latency is not a constraint.** 15.4 ms mean per horizon, 28.8 ms worst,
74 ms for all six horizons together, against a 200 ms SLA. The cost is
dominated by the baseline-ladder merge over the history window, not by model
inference (~0.01 ms/row batched).

**Shadow deployment works, and the snapshot is interesting.** Production
serves the baseline ensemble; the candidate runs alongside on 40 labelled
stations and both are logged:

| horizon | production MAE | candidate MAE | candidate better |
|---|---|---|---|
| 1 h | 22.16 | 19.61 | yes |
| 3 h | 21.95 | 22.91 | no |
| 6 h | 6.64 | 7.59 | no |
| 12 h | 22.65 | 17.67 | yes |
| 24 h | 13.60 | 10.27 | yes |
| 48 h | 18.98 | 12.26 | yes |

The candidate wins 4 of 6. **This is 40 stations at one timestamp and it is
not evidence of anything** — the sample is far too small and the horizons
disagree with each other. It is here to show the mechanism runs end to end
and that both forecasts land in a comparable log. A real shadow evaluation
accumulates over weeks; that loop does not exist yet.

**Drift references are stored** for ten input distributions and the
prediction distribution per horizon. Two numbers worth carrying into
monitoring: `regional_pm25_mean_50km` and `upwind_pm25_mean` are 34-36% null
in training, so an alert on their null rate needs that as its baseline, not
zero. And predicted means (26.6-28.3) track actual means (27.8-28.2) closely
while p95 of predictions (44.8-60.0) sits far below the data's — the
tail-compression from notebook 07, visible directly in the output
distribution.

**What production would serve today: the baseline ensemble, on every
horizon, with `fallback_used: true` and a documented reason.** That is the
system behaving correctly. It is also the honest summary of this pipeline —
the machinery is finished and the model has not yet earned the right to use
it.